# Clean Supplied Product Reviews and Export CSV

This notebook prepares the user-provided review records and saves `product_reviews_clean.csv` with exactly one column: `review_text`. It normalizes whitespace, validates the supplied data, and preserves the original review wording.

The supplied data contains 93 reviews (IDs 1–93), fewer than the 100-review collection target. The notebook preserves the supplied records and reports the shortfall; it does not invent or duplicate reviews. The records are user-provided and are not claimed to have been independently scraped.

In [33]:
from io import StringIO
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

raw_reviews = '''review_id,product_category,rating,review_text,verified_purchase
1,Electronics,5,"Excellent battery life and the active noise cancellation works beautifully on my daily train commute. Worth the price.",True
2,Electronics,1,"Stopped charging after exactly 33 days. Just missed the return window. Completely dead, do not buy!!",True
3,Electronics,4,"sound quality is crisp but the ear cups are a bit tight if you wear glasses. still keeping it though.",True
4,Electronics,2,"The mic quality is garbage. Everyone on my Zoom calls says I sound like I am underwater.",False
5,Electronics,5,"Blown away by the bass response. Connects instantly to both my iPhone and laptop without manual switching.",True
6,Electronics,3,"It's just okay. Nothing special for the money. The plastic frame feels a bit flimsy.",True
7,Electronics,1,"Disappointed. Constant Bluetooth dropping issues. Had to restart my phone multiple times to sync.",True
8,Electronics,5,"Best purchase of the year. Battery lasts almost a whole week of moderate use.",True
9,Electronics,4,"Good headphones overall but the charging cable included in the box is ridiculously short.",True
10,Electronics,2,"Too heavy on the head. Gives me a headache after an hour of gaming.",False
11,Apparel,5,"Fits perfectly! The material is thick, breathable, and doesn't shrink in the washer. Will order more colors.",True
12,Apparel,1,"Sizing is completely off. Ordered a Large but it fits like a Small. Material feels rough and cheap.",True
13,Apparel,4,"Nice hoodie. Very soft inside. Lost one star because the zipper gets stuck occasionally.",True
14,Apparel,3,"Color doesn't look like the photo online. It's more of a dull gray than navy blue.",True
15,Apparel,5,"Super comfortable for working out. Sweat-wicking works as advertised.",False
16,Apparel,2,"Stitching started coming apart at the seams after the first wash. Poor quality control.",True
17,Apparel,4,"A bit baggy but very warm. Great for lounging around the house.",True
18,Apparel,1,"Arrived with a weird chemical smell that didn't go away even after washing it twice.",True
19,Apparel,5,"Excellent quality for the price point. Highly recommend checking the size chart before buying.",True
20,Apparel,3,"Decent shirt but shrinks significantly if you accidentally put it in the dryer.",True
21,Home & Kitchen,5,"This blender is a beast. Crushes ice into snow in seconds. Smoothies come out perfectly consistent.",True
22,Home & Kitchen,1,"Leaked from the bottom container on the very first use. Made a massive mess on my counter.",True
23,Home & Kitchen,4,"Great coffee maker, brews super fast. Wish the water reservoir was slightly larger.",True
24,Home & Kitchen,2,"Extremely loud. Sounds like a lawnmower is running in my kitchen every morning.",False
25,Home & Kitchen,5,"Easy to clean, compact design, and looks sleek on the counter. Very satisfied.",True
26,Home & Kitchen,3,"Works fine for basic things but struggles with frozen fruits or tougher vegetables.",True
27,Home & Kitchen,1,"The heating element burnt out after two months. Customer service refused to replace it.",True
28,Home & Kitchen,4,"Saves so much time cooking dinner. Non-stick coating seems durable so far.",True
29,Home & Kitchen,5,"Absolute game changer for meal prep. Don't know how I lived without it.",True
30,Home & Kitchen,2,"Instruction manual is poorly translated and impossible to follow. Had to look up YouTube tutorials.",False
31,Beauty,5,"My skin has never looked better. Hydrating without feeling greasy or causing breakouts.",True
32,Beauty,1,"Broke me out in hives within 24 hours. Beware if you have sensitive skin.",True
33,Beauty,4,"Smells amazing and leaves hair soft, but the pump bottle mechanism arrived broken.",True
34,Beauty,3,"Average product. Didn't notice any dramatic change after using the entire bottle.",True
35,Beauty,5,"A little goes a long way. This bottle will easily last me six months.",True
36,Beauty,2,"Way too drying for my skin type. Made my foundation look patchy.",False
37,Beauty,4,"Good value for money. Texture is lightweight and absorbs quickly.",True
38,Beauty,1,"Fake product! Packaging looks completely different from the one I bought at Sephora.",False
39,Beauty,5,"Highly recommend! Cleared up my dark spots in less than three weeks.",True
40,Beauty,3,"The scent is overpowering. If you are sensitive to strong perfumes, skip this.",True
41,Sports & Outdoors,5,"Keeps water ice cold even after sitting in a hot car for 8 hours. Best flask ever.",True
42,Sports & Outdoors,1,"The zipper on the backpack ripped completely open during my first hike. Not heavy duty at all.",True
43,Sports & Outdoors,4,"Very lightweight and easy to fold up. Great additions to our camping gear setup.",True
44,Sports & Outdoors,2,"The resistance bands snapped after a week of standard use. Dangerous design.",True
45,Sports & Outdoors,5,"Excellent grip on wet trails. No slipping or blisters after a 10-mile break-in run.",True
46,Sports & Outdoors,3,"Good mat but it stretches out a bit during hot yoga sessions.",True
47,Sports & Outdoors,1,"Missing half the parts in the box. Couldn't even assemble the net.",True
48,Sports & Outdoors,4,"Solid basketball for outdoor courts. Holds air pressure well over time.",True
49,Sports & Outdoors,5,"Tracking accuracy on this watch is spot on compared to my older premium devices.",False
50,Sports & Outdoors,2,"The fabric is thin and feels like it will rip if it catches on any branch.",True
51,Electronics,5,"The display is gorgeous. Refresh rate makes scrolling incredibly fluid and responsive.",True
52,Electronics,1,"Screen flickering issue out of the box. Returning this immediately.",True
53,Electronics,4,"Decent laptop for students. Lightweight, good trackpad, but speakers are quiet.",True
54,Electronics,3,"Gets incredibly hot while playing basic indie games. Fan sounds like a jet engine.",False
55,Electronics,5,"Unbelievable value. Performance rivals tablets that cost twice as much.",True
56,Electronics,2,"The user interface is laggy and full of pre-installed bloatware apps you can't delete.",True
57,Electronics,4,"Fast charging works great. Goes from 0 to 80% in about 35 minutes.",True
58,Electronics,1,"Do not buy refurbished from here. Scratches all over the screen and battery health is at 60%.",True
59,Electronics,5,"Setup took less than 2 minutes. Strong mesh Wi-Fi signal across my entire house now.",True
60,Electronics,3,"The remote control requires exact line of sight or it won't register inputs.",True
61,Apparel,5,"These jeans have the perfect amount of stretch. Comfortable all day long.",True
62,Apparel,1,"The color bled all over my other clothes in the laundry despite washing with cold water.",True
63,Apparel,4,"Warm winter coat. Plenty of pockets. It runs slightly larger than standard sizing.",True
64,Apparel,2,"The material is incredibly sheer. You can see right through it in sunlight.",False
65,Apparel,5,"Classic design, well made, and fits true to size. Will buy from this brand again.",True
66,Apparel,3,"Socks are fine but elastic around the ankle is loose and they slide down.",True
67,Apparel,1,"Button fell off within ten minutes of wearing it for the first time. Terrible stitching.",True
68,Apparel,4,"Great summer dress. Lightweight and breathable. Needs an iron out of the package.",True
69,Apparel,5,"Feels like premium designer quality without the insane luxury price markup.",True
70,Apparel,2,"Shrunk two whole sizes after one gentle cycle wash. Hand wash only I guess.",True
71,Home & Kitchen,5,"Super sharp knives. Cut through raw meat like butter. Handle balance feels great.",True
72,Home & Kitchen,1,"The nonstick pan started peeling off flecks into our food after three weeks.",True
73,Home & Kitchen,4,"Air fryer works incredibly well. Crispy fries without the oil. Basket cleaning is easy.",True
74,Home & Kitchen,3,"Toasts unevenly. One side of the bread is burnt while the other is barely warm.",True
75,Home & Kitchen,5,"Organized my entire pantry with these containers. Airtight seal keeps cereal fresh.",True
76,Home & Kitchen,2,"The vacuum cleaner suction dropped significantly after cleaning up a small patch of pet hair.",True
77,Home & Kitchen,4,"Good basic microwave. Controls are intuitive. A bit louder than my previous one.",True
78,Home & Kitchen,1,"The glass pitcher shattered when pouring warm tea. Absolute safety hazard.",False
79,Home & Kitchen,5,"Saves so much space in my tiny kitchen. Foldable design is genius.",True
80,Home & Kitchen,3,"It gets the job done but the power cord is too short to reach my island outlet.",True
81,Beauty,5,"This lip balm is the only thing that saves my dry chapped lips during winter.",True
82,Beauty,1,"Gave me a massive rash. Check the ingredient list carefully before purchasing.",True
83,Beauty,4,"Matte finish stays on all day without smudging, but it's hard to remove at night.",True
84,Beauty,2,"The product arrived completely dried out and unusable. Seemed like old stock.",True
85,Beauty,5,"Waterproof formula actually holds up during swimming and heavy gym sessions.",True
86,Beauty,3,"Decent coverage but the shade range is very limited. Hard to match neutral undertones.",False
87,Beauty,4,"Leaves hair looking shiny and bouncy without weighing it down. Subtle clean scent.",True
88,Beauty,1,"Extremely greasy formula. Left my skin looking like an oil slick and clogged my pores.",True
89,Beauty,5,"Perfect dupe for the expensive high-end department store serums. Don't overpay.",True
90,Beauty,3,"The dropper bottle leaks from the top cap if it tips over in your makeup bag.",True
91,Sports & Outdoors,5,"Very stable kayak even in windy lake conditions. Easy to inflate and pack away.",True
92,Sports & Outdoors,1,"The pedal fell completely off the bike crank arm while riding. Threading was stripped.",True
93,Sports & Outdoors,4,"Excellent yoga blocks. Dense foam provides reliable support without compressing.",True
'''

reviews_df = pd.read_csv(StringIO(raw_reviews), dtype={"product_category": "string"})
reviews_df.columns = [column.strip().lower() for column in reviews_df.columns]
reviews_df["review_text"] = reviews_df["review_text"].fillna("").map(
    lambda text: re.sub(r"\s+", " ", str(text)).strip()
)
reviews_df["product_category"] = reviews_df["product_category"].str.strip()
reviews_df["rating"] = pd.to_numeric(reviews_df["rating"], errors="raise").astype("int64")
reviews_df["verified_purchase"] = (
    reviews_df["verified_purchase"].astype("string").str.strip().str.lower().map(
        {"true": True, "false": False}
    )
)

expected_columns = ["review_id", "product_category", "rating", "review_text", "verified_purchase"]
assert list(reviews_df.columns) == expected_columns
assert reviews_df["review_id"].is_unique
assert reviews_df["review_text"].notna().all()
assert reviews_df["review_text"].str.len().gt(0).all()
assert reviews_df["rating"].between(1, 5).all()
assert reviews_df["verified_purchase"].notna().all()

output_dir = Path.cwd()
if output_dir.name.lower() != "assignment1":
    candidate_dir = output_dir / "assignments" / "assignment1"
    if candidate_dir.is_dir():
        output_dir = candidate_dir
output_csv = output_dir / "product_reviews_clean.csv"
reviews_df.to_csv(output_csv, index=False, encoding="utf-8")

print(f"Saved {len(reviews_df)} cleaned records to {output_csv.resolve()}")
if len(reviews_df) < 100:
    print(f"Note: the supplied data has {len(reviews_df)} reviews; 100 were requested.")
display(reviews_df.head())

Saved 93 cleaned records to E:\KrishNaik\learning_workspace\assignments\assignment1\product_reviews_clean.csv
Note: the supplied data has 93 reviews; 100 were requested.


,review_id,product_category,rating,review_text,verified_purchase
0,1,Electronics,5,Excellent battery life and the active noise ca...,True
1,2,Electronics,1,Stopped charging after exactly 33 days. Just m...,True
2,3,Electronics,4,sound quality is crisp but the ear cups are a ...,True
3,4,Electronics,2,The mic quality is garbage. Everyone on my Zoo...,False
4,5,Electronics,5,Blown away by the bass response. Connects inst...,True


In [34]:
# Export only the review text column requested for the final dataset.
review_text_df = reviews_df[["review_text"]].copy()
review_text_df.to_csv(output_csv, index=False, encoding="utf-8")

saved_reviews = pd.read_csv(output_csv)
assert list(saved_reviews.columns) == ["review_text"]
assert len(saved_reviews) == len(reviews_df)
assert saved_reviews["review_text"].notna().all()
assert saved_reviews["review_text"].str.strip().ne("").all()
print(f"Saved {len(saved_reviews)} review texts to {output_csv.resolve()}")

Saved 93 review texts to E:\KrishNaik\learning_workspace\assignments\assignment1\product_reviews_clean.csv


## Task 1: Text Preprocessing

Run these cells in order. They work on a separate `preprocessing_df` copied from the review-text-only CSV data and do not change the original review text. Tokenization uses whitespace splitting; punctuation is removed in its own following step. Stopword filtering removes `and`, `a`, and `the`. WordNet lemmatization is optional and requires the NLTK WordNet corpus.

In [35]:
# Step 1: Convert review text to lowercase.
preprocessing_df = saved_reviews.copy()
preprocessing_df["review_text"] = preprocessing_df["review_text"].str.lower()
display(preprocessing_df[["review_text"]].head())

,review_text
0,excellent battery life and the active noise ca...
1,stopped charging after exactly 33 days. just m...
2,sound quality is crisp but the ear cups are a ...
3,the mic quality is garbage. everyone on my zoo...
4,blown away by the bass response. connects inst...


In [36]:
# Step 2: Tokenize each review by whitespace.
preprocessing_df["tokens"] = preprocessing_df["review_text"].str.split()
display(preprocessing_df[["tokens"]].head())

,tokens
0,"[excellent, battery, life, and, the, active, n..."
1,"[stopped, charging, after, exactly, 33, days.,..."
2,"[sound, quality, is, crisp, but, the, ear, cup..."
3,"[the, mic, quality, is, garbage., everyone, on..."
4,"[blown, away, by, the, bass, response., connec..."


In [37]:
# Step 3: Remove punctuation from each token and discard empty tokens.
import string

punctuation_table = str.maketrans("", "", string.punctuation)
preprocessing_df["tokens"] = preprocessing_df["tokens"].apply(
    lambda tokens: [token.translate(punctuation_table) for token in tokens]
)
preprocessing_df["tokens"] = preprocessing_df["tokens"].apply(
    lambda tokens: [token for token in tokens if token]
)
display(preprocessing_df[["tokens"]].head())

,tokens
0,"[excellent, battery, life, and, the, active, n..."
1,"[stopped, charging, after, exactly, 33, days, ..."
2,"[sound, quality, is, crisp, but, the, ear, cup..."
3,"[the, mic, quality, is, garbage, everyone, on,..."
4,"[blown, away, by, the, bass, response, connect..."


In [38]:
# Step 4 (optional): Remove the requested stopwords.
stopwords = {"and", "a", "the"}
preprocessing_df["tokens"] = preprocessing_df["tokens"].apply(
    lambda tokens: [token for token in tokens if token not in stopwords]
)
display(preprocessing_df[["tokens"]].head())

,tokens
0,"[excellent, battery, life, active, noise, canc..."
1,"[stopped, charging, after, exactly, 33, days, ..."
2,"[sound, quality, is, crisp, but, ear, cups, ar..."
3,"[mic, quality, is, garbage, everyone, on, my, ..."
4,"[blown, away, by, bass, response, connects, in..."


In [39]:
# Step 5 (optional): Lemmatize tokens with WordNet, when its corpus is installed.
import nltk
from nltk.stem import WordNetLemmatizer

try:
    nltk.data.find("corpora/wordnet")
except LookupError:
    preprocessing_df["lemmatized_tokens"] = preprocessing_df["tokens"].copy()
    print("WordNet corpus is not installed; lemmatization was skipped. Run nltk.download('wordnet') to enable it.")
else:
    lemmatizer = WordNetLemmatizer()
    preprocessing_df["lemmatized_tokens"] = preprocessing_df["tokens"].apply(
        lambda tokens: [lemmatizer.lemmatize(token) for token in tokens]
    )

display(preprocessing_df[["review_text", "tokens", "lemmatized_tokens"]].head())

WordNet corpus is not installed; lemmatization was skipped. Run nltk.download('wordnet') to enable it.


,review_text,tokens,lemmatized_tokens
0,excellent battery life and the active noise ca...,"[excellent, battery, life, active, noise, canc...","[excellent, battery, life, active, noise, canc..."
1,stopped charging after exactly 33 days. just m...,"[stopped, charging, after, exactly, 33, days, ...","[stopped, charging, after, exactly, 33, days, ..."
2,sound quality is crisp but the ear cups are a ...,"[sound, quality, is, crisp, but, ear, cups, ar...","[sound, quality, is, crisp, but, ear, cups, ar..."
3,the mic quality is garbage. everyone on my zoo...,"[mic, quality, is, garbage, everyone, on, my, ...","[mic, quality, is, garbage, everyone, on, my, ..."
4,blown away by the bass response. connects inst...,"[blown, away, by, bass, response, connects, in...","[blown, away, by, bass, response, connects, in..."


In [40]:
# Vocabulary size and top word frequencies with scikit-learn.
from sklearn.feature_extraction.text import CountVectorizer

word_column = "lemmatized_tokens" if "lemmatized_tokens" in preprocessing_df.columns else "tokens"
documents = preprocessing_df[word_column].map(" ".join)
vectorizer = CountVectorizer(token_pattern=r"(?u)\b\w+\b", lowercase=False)
word_matrix = vectorizer.fit_transform(documents)

vocabulary_size = len(vectorizer.vocabulary_)
word_frequencies = pd.DataFrame(
    {
        "word": vectorizer.get_feature_names_out(),
        "frequency": word_matrix.sum(axis=0).A1,
    }
).sort_values(["frequency", "word"], ascending=[False, True])

print(f"Vocabulary size: {vocabulary_size}")
print("Top 20 frequent words:")
display(word_frequencies.head(20).reset_index(drop=True))

Vocabulary size: 694
Top 20 frequent words:


,word,frequency
0,is,24
1,my,21
2,to,17
3,it,16
4,in,15
5,of,14
6,but,13
7,for,13
8,after,12
9,on,12


## Task 3: Feature Engineering

Run these cells after text preprocessing. Each cell builds a document-term representation from the processed tokens and prints its dimensions with a compact example.

In [41]:
# 1. One-hot encoding: binary word presence for each review document.
from sklearn.preprocessing import MultiLabelBinarizer

one_hot_encoder = MultiLabelBinarizer(sparse_output=True)
one_hot_matrix = one_hot_encoder.fit_transform(preprocessing_df["tokens"])
print(f"One-hot document-term matrix shape: {one_hot_matrix.shape}")
first_document_presence = pd.Series(
    one_hot_matrix[0].toarray().ravel(),
    index=one_hot_encoder.classes_,
    name="present",
)
print("Words present in the first review:")
display(first_document_presence[first_document_presence.eq(1)].head(20).to_frame())

One-hot document-term matrix shape: (93, 694)
Words present in the first review:


,present
active,1
battery,1
beautifully,1
cancellation,1
commute,1
daily,1
excellent,1
life,1
my,1
noise,1


In [44]:
# 2. Bag of Words: raw token counts per review document.
from sklearn.feature_extraction.text import CountVectorizer

feature_documents = preprocessing_df["tokens"].map(" ".join)
bow_vectorizer = CountVectorizer(token_pattern=r"(?u)\b\w+\b", lowercase=False)
bow_matrix = bow_vectorizer.fit_transform(feature_documents)
print(f"Bag-of-Words matrix shape: {bow_matrix.shape}")
print(f"Bag-of-Words vocabulary size: {len(bow_vectorizer.vocabulary_)}")

top_bow_terms = pd.DataFrame(
    {
        "word": bow_vectorizer.get_feature_names_out(),
        "count": bow_matrix.sum(axis=0).A1,
    }
).sort_values(["count", "word"], ascending=[False, True]).head(20)
display(top_bow_terms.reset_index(drop=True))

print(bow_matrix.toarray())

Bag-of-Words matrix shape: (93, 694)
Bag-of-Words vocabulary size: 694


,word,count
0,is,24
1,my,21
2,to,17
3,it,16
4,in,15
5,of,14
6,but,13
7,for,13
8,after,12
9,on,12


[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [43]:
# 3. TF-IDF: term frequency weighted by inverse document frequency.
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b", lowercase=False)
tfidf_matrix = tfidf_vectorizer.fit_transform(feature_documents)
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

first_document_tfidf = pd.Series(
    tfidf_matrix[0].toarray().ravel(),
    index=tfidf_vectorizer.get_feature_names_out(),
)
print("Highest-weight terms in the first review:")
display(
    first_document_tfidf[first_document_tfidf.gt(0)]
    .sort_values(ascending=False)
    .head(10)
    .rename("tfidf_weight")
    .to_frame()
)

TF-IDF matrix shape: (93, 694)
Highest-weight terms in the first review:


,tfidf_weight
active,0.283732
beautifully,0.283732
cancellation,0.283732
commute,0.283732
daily,0.283732
life,0.283732
noise,0.283732
train,0.283732
worth,0.283732
battery,0.243183


## Task 4: Comparison Analysis

Compare the document-level representations and inspect the terms with the highest mean TF-IDF weight across reviews.

In [47]:
# Compare representation behavior and identify informative TF-IDF terms.
import numpy as np

comparison_table = pd.DataFrame(
    [
        {
            "Method": "One-Hot Encoding",
            "Matrix shape": one_hot_matrix.shape,
            "Values": "0 or 1",
            "Meaning": "Whether a word appears in a review",
        },
        {
            "Method": "Bag of Words",
            "Matrix shape": bow_matrix.shape,
            "Values": "Non-negative integer counts",
            "Meaning": "How many times a word appears in a review",
        },
        {
            "Method": "TF-IDF",
            "Matrix shape": tfidf_matrix.shape,
            "Values": "Non-negative weighted scores",
            "Meaning": "Term frequency adjusted by rarity across reviews",
        },
    ]
)
display(comparison_table)

# Review-level TF-IDF highlights terms important to the first review.
first_review_scores = pd.Series(
    tfidf_matrix[0].toarray().ravel(),
    index=tfidf_vectorizer.get_feature_names_out(),
)
print("Highest-weight TF-IDF terms in the first review:")
display(
    first_review_scores[first_review_scores.gt(0)]
    .sort_values(ascending=False)
    .head(10)
    .rename("tfidf_weight")
    .to_frame()
)

# High IDF indicates that a term occurs in relatively few reviews.
document_frequency = np.asarray((bow_matrix > 0).sum(axis=0)).ravel()
idf_analysis = pd.DataFrame(
    {
        "word": tfidf_vectorizer.get_feature_names_out(),
        "reviews_containing_word": document_frequency,
        "idf": tfidf_vectorizer.idf_,
    }
)
idf_analysis = idf_analysis[idf_analysis["word"].str.isalpha()]
idf_analysis = idf_analysis.sort_values(["idf", "word"], ascending=[False, True])
print("Rarest alphabetic words with the highest IDF across the dataset:")
display(idf_analysis.head(10).reset_index(drop=True))
print(
    "TF-IDF combines a term's frequency within one review with its inverse document frequency. "
    "With scikit-learn's default smoothing, IDF = log((1 + number_of_reviews) / "
    "(1 + reviews_containing_word)) + 1. A common word appears in many reviews, so its IDF "
    "multiplier is lower; rarer, review-specific words receive a higher IDF. Only 'and', 'a', "
    "and 'the' were removed as stopwords, so other common words remain but are down-weighted."
)

,Method,Matrix shape,Values,Meaning
0,One-Hot Encoding,"(93, 694)",0 or 1,Whether a word appears in a review
1,Bag of Words,"(93, 694)",Non-negative integer counts,How many times a word appears in a review
2,TF-IDF,"(93, 694)",Non-negative weighted scores,Term frequency adjusted by rarity across reviews


Highest-weight TF-IDF terms in the first review:


,tfidf_weight
active,0.283732
beautifully,0.283732
cancellation,0.283732
commute,0.283732
daily,0.283732
life,0.283732
noise,0.283732
train,0.283732
worth,0.283732
battery,0.243183


Rarest alphabetic words with the highest IDF across the dataset:


,word,reviews_containing_word,idf
0,about,1,4.850148
1,absorbs,1,4.850148
2,accidentally,1,4.850148
3,accuracy,1,4.850148
4,across,1,4.850148
5,active,1,4.850148
6,actually,1,4.850148
7,additions,1,4.850148
8,advertised,1,4.850148
9,again,1,4.850148


TF-IDF combines a term's frequency within one review with its inverse document frequency. With scikit-learn's default smoothing, IDF = log((1 + number_of_reviews) / (1 + reviews_containing_word)) + 1. A common word appears in many reviews, so its IDF multiplier is lower; rarer, review-specific words receive a higher IDF. Only 'and', 'a', and 'the' were removed as stopwords, so other common words remain but are down-weighted.


## Task 5: Sparse Matrix Analysis

Inspect the shape and zero percentage of each feature matrix without converting it to a dense array.

In [48]:
# Report sparsity directly from sparse matrices; do not call toarray() here.
feature_matrices = {
    "One-Hot Encoding": one_hot_matrix,
    "Bag of Words": bow_matrix,
    "TF-IDF": tfidf_matrix,
}

sparsity_rows = []
for method_name, matrix in feature_matrices.items():
    total_entries = matrix.shape[0] * matrix.shape[1]
    zero_entries = total_entries - matrix.nnz
    zero_percentage = (zero_entries / total_entries * 100) if total_entries else 0.0
    sparsity_rows.append(
        {
            "Method": method_name,
            "Shape (documents, terms)": matrix.shape,
            "Stored non-zero values": matrix.nnz,
            "Zero values": zero_entries,
            "Sparsity (%)": round(zero_percentage, 2),
        }
    )

sparsity_table = pd.DataFrame(sparsity_rows)
display(sparsity_table)

print(
    "Sparse matrices are usually efficient for large text vocabularies when most entries are zero: "
    "they store non-zero values plus index information instead of every zero. Sparse formats can "
    "become less efficient when the matrix is dense, because index overhead and indirect access "
    "may cost more than dense storage. Some algorithms also do not support sparse input. Avoid "
    "converting large matrices with toarray(), which allocates storage for every document-term cell "
    "and can exhaust memory."
)

,Method,"Shape (documents, terms)",Stored non-zero values,Zero values,Sparsity (%)
0,One-Hot Encoding,"(93, 694)",1153,63389,98.21
1,Bag of Words,"(93, 694)",1153,63389,98.21
2,TF-IDF,"(93, 694)",1153,63389,98.21


Sparse matrices are usually efficient for large text vocabularies when most entries are zero: they store non-zero values plus index information instead of every zero. Sparse formats can become less efficient when the matrix is dense, because index overhead and indirect access may cost more than dense storage. Some algorithms also do not support sparse input. Avoid converting large matrices with toarray(), which allocates storage for every document-term cell and can exhaust memory.


## Task 6: Real-World Questions

### 1. Why Bag of Words does not understand semantic meaning

Bag of Words represents each review using token counts. It recognizes that two reviews share a word, but it does not know that different words or phrases can express similar ideas. For example, review 4 says the mic quality is “garbage,” while review 16 describes “poor quality control.” Both are negative quality comments, but their vocabularies overlap little, so a basic BoW model will not recognize their shared meaning without enough labeled examples or additional features.

BoW also ignores word order and often misses negation. In this dataset, “Best purchase of the year” (review 8) is positive, while “do not buy” (review 2) is negative. A unigram representation counts words independently and does not inherently understand how “not” changes the meaning of “buy.” Adding n-grams can capture some local phrases, but it still does not provide general semantic understanding.

### 2. When to use Bag of Words and TF-IDF in industry

- **Bag of Words:** Use as a simple, interpretable baseline for small or medium text-classification tasks, such as routing support tickets, categorizing reviews, or detecting a known set of keywords. It works well with linear models and Naive Bayes when the signal is strongly tied to particular words or phrases.
- **TF-IDF:** Use when words should be weighted by how distinctive they are across a collection. Typical uses include lexical search and document retrieval, ranking relevant support articles, grouping similar documents, and text classification where common terms should count less than discriminative terms.
- Both are useful when speed, interpretability, and a small operational footprint matter. Neither should be treated as a semantic embedding model.

### 3. Limitations of TF-IDF in real applications

- It is based on token overlap, not meaning. “Garbage” and “poor” may both signal dissatisfaction, but TF-IDF does not know they are related.
- It does not inherently capture word order or negation. “Good” and “not good” can receive similar lexical features unless phrase n-grams are configured; even n-grams only capture limited context.
- It creates a large, sparse vocabulary. New product terms, misspellings, slang, and unseen words are not represented usefully until the vocabulary is updated.
- Its IDF weights depend on the collection used to fit it. A change in product mix or language can change which terms are considered distinctive, so production systems may need periodic refitting.
- Rare typos or accidental tokens can receive high IDF despite having little practical importance. High TF-IDF is a statistical signal of distinctiveness, not a guarantee that a word is meaningful.

For example, “battery” appears in both positive and negative reviews here: review 8 praises battery life, while review 58 reports poor battery health. TF-IDF can weight the word according to frequency and corpus rarity, but it cannot infer the different sentiment or underlying product condition from that word alone.

## Task 7: Mini Use Case - Sentiment Classification

Treat 4- and 5-star reviews as positive, 1- and 2-star reviews as negative, and exclude neutral 3-star reviews. Both models use the same stratified train/test split. The vectorizers are fit inside each pipeline using training data only.

In [49]:
# Prepare binary sentiment labels and a shared stratified holdout split.
from sklearn.model_selection import train_test_split

sentiment_data = reviews_df.loc[reviews_df["rating"].ne(3), ["review_text", "rating"]].copy()
sentiment_data["sentiment"] = sentiment_data["rating"].map(
    lambda rating: "positive" if rating >= 4 else "negative"
)

X_train_text, X_test_text, y_train, y_test = train_test_split(
    sentiment_data["review_text"],
    sentiment_data["sentiment"],
    test_size=0.25,
    random_state=42,
    stratify=sentiment_data["sentiment"],
)
sentiment_results = {}

print(f"Usable reviews: {len(sentiment_data)} of {len(reviews_df)} (neutral 3-star reviews excluded)")
print(f"Training rows: {len(X_train_text)}; test rows: {len(X_test_text)}")
print("Class counts:")
display(sentiment_data["sentiment"].value_counts().rename_axis("sentiment").to_frame("reviews"))

Usable reviews: 79 of 93 (neutral 3-star reviews excluded)
Training rows: 59; test rows: 20
Class counts:


,reviews
sentiment,
positive,47
negative,32


In [50]:
# Fit Logistic Regression with Bag-of-Words features.
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

bow_sentiment_model = Pipeline(
    [
        ("vectorizer", CountVectorizer(ngram_range=(1, 2))),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]
)
bow_sentiment_model.fit(X_train_text, y_train)
bow_predictions = bow_sentiment_model.predict(X_test_text)
sentiment_results["BoW + Logistic Regression"] = {
    "Accuracy": accuracy_score(y_test, bow_predictions),
    "Precision (positive)": precision_score(y_test, bow_predictions, pos_label="positive", zero_division=0),
    "Recall (positive)": recall_score(y_test, bow_predictions, pos_label="positive", zero_division=0),
    "F1 (positive)": f1_score(y_test, bow_predictions, pos_label="positive", zero_division=0),
}
print("Bag-of-Words Logistic Regression fitted and evaluated.")

Bag-of-Words Logistic Regression fitted and evaluated.


In [51]:
# Fit Logistic Regression with TF-IDF features on the same split.
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_sentiment_model = Pipeline(
    [
        ("vectorizer", TfidfVectorizer(ngram_range=(1, 2))),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]
)
tfidf_sentiment_model.fit(X_train_text, y_train)
tfidf_predictions = tfidf_sentiment_model.predict(X_test_text)
sentiment_results["TF-IDF + Logistic Regression"] = {
    "Accuracy": accuracy_score(y_test, tfidf_predictions),
    "Precision (positive)": precision_score(y_test, tfidf_predictions, pos_label="positive", zero_division=0),
    "Recall (positive)": recall_score(y_test, tfidf_predictions, pos_label="positive", zero_division=0),
    "F1 (positive)": f1_score(y_test, tfidf_predictions, pos_label="positive", zero_division=0),
}
print("TF-IDF Logistic Regression fitted and evaluated.")

TF-IDF Logistic Regression fitted and evaluated.


In [52]:
# Compare the held-out performance of both feature representations.
sentiment_comparison = pd.DataFrame(sentiment_results).T
sentiment_comparison.index.name = "Model"
display(sentiment_comparison.round(3))

print(
    "Interpret these scores cautiously: the dataset is small and uses one stratified holdout split. "
    "Use more labeled reviews and stratified cross-validation before selecting a production model."
)

,Accuracy,Precision (positive),Recall (positive),F1 (positive)
Model,,,,
BoW + Logistic Regression,0.75,0.733,0.917,0.815
TF-IDF + Logistic Regression,0.80,0.900,0.750,0.818


Interpret these scores cautiously: the dataset is small and uses one stratified holdout split. Use more labeled reviews and stratified cross-validation before selecting a production model.
